# 02. Practice: 비용, 출력, 강건성 toy 계산

## 목표

- KV 캐시 증가를 숫자로 확인합니다.
- 통합 생성형 비전 출력 스키마를 검증합니다.
- 텍스트 오정보와 이미지 증거가 충돌할 때 간단한 점수 규칙을 만들어 봅니다.


In [ ]:
def kv_cache_gib(layers, hidden_size, seq_len, bytes_per_value=2):
    """KV 캐시를 GiB로 근사합니다. 실제 구현보다 단순하지만 길이 증가 효과를 보기 충분합니다."""
    return (2 * layers * hidden_size * seq_len * bytes_per_value) / (1024 ** 3)

for seq_len in [1024, 4096, 8192, 32768]:
    print(seq_len, f"{kv_cache_gib(32, 4096, seq_len):.2f} GiB")


In [ ]:
def full_attention_tokens(reference_tokens, generated_tokens):
    return reference_tokens + generated_tokens

def rswa_tokens(reference_tokens, generated_tokens, window):
    return reference_tokens + min(generated_tokens, window)

reference = 512
window = 256
for generated in [256, 1024, 4096, 32768]:
    print(generated, full_attention_tokens(reference, generated), rswa_tokens(reference, generated, window))


In [ ]:
import json

def validate_multimodal_output(text):
    """생성 모델 출력이 최소 스키마를 만족하는지 검사합니다."""
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        return False, "not json"
    task = obj.get("task")
    if task == "detect":
        return isinstance(obj.get("objects"), list), "detect requires objects"
    if task == "depth":
        return isinstance(obj.get("map"), list), "depth requires map"
    if task == "pose":
        return all(k in obj for k in ["translation", "rotation"]), "pose requires translation and rotation"
    return False, "unknown task"

examples = [
    '{"task":"detect","objects":[{"label":"table","bbox":[1,2,3,4]}]}',
    '{"task":"pose","translation":[0,0,1],"rotation":[0,0,0,1]}',
    'detect the table at 1 2 3 4',
]
for example in examples:
    print(validate_multimodal_output(example))


In [ ]:
def robust_decision(image_score, text_score, conflict):
    """이미지와 텍스트가 충돌하면 이미지 증거에 가중치를 더 주는 toy rule입니다."""
    image_weight = 0.75 if conflict else 0.5
    text_weight = 1.0 - image_weight
    return image_score * image_weight + text_score * text_weight

cases = [
    {"image": 0.9, "text": 0.2, "conflict": True},
    {"image": 0.6, "text": 0.7, "conflict": False},
    {"image": 0.4, "text": 0.95, "conflict": True},
]
for case in cases:
    print(case, "->", round(robust_decision(case["image"], case["text"], case["conflict"]), 3))
